# 第一阶段：建立全景图

这个阶段不学 CUDA、不读 vLLM 源码、不追论文细节，只建立一个可靠的“大模型推理心智模型”。

最终成果不是“看完几节课”，而是一个小型实验仓库：
> 对一个小模型完成推理链路分析，测量 TTFT、TPOT、吞吐和显存，并解释 KV Cache、Prefill、Decode 与 Batch 的关系。

**两周后的通关目标：**
你应该能独立回答：
1. model.generate() 内部大致发生了什么？
2. Prefill 和 Decode 分别在计算什么？
3. KV Cache 保存了什么，为什么能加速生成？
4. 为什么 KV Cache 会随并发量和上下文长度快速增长？
5. 为什么 Prefill 和 Decode 的性能特征不同？
6. 吞吐、TTFT、TPOT、P50/P99 分别代表什么？
7. 为什么 batch 增大通常提高吞吐，却可能恶化单请求延迟？
8. 为什么 benchmark 必须 warmup 和同步 GPU？
9. 为什么一次测试结果不能直接得出性能结论？
并且能够拿出自己的实验数据，而不是只复述概念。

## 一、先跑起来，再理解
> 目标：知道模型输入输出的基本形态。
> 完成：
> 1. 加载 tokenizer 和 model；
> 2. 输入一段固定文本；
> 3. 调用 generate()；
> 4. 打印 token IDs；
> 5. 打印每个 token 对应的文本；
> 6. 输出输入 token 数和生成 token 数。

### 0.准备工作

In [1]:
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

# 加载分词器和模型
MODEL_PATH = Path("../../models/Qwen2.5-1.5B-Instruct")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="cuda"
)

# 准备测试文本及分词
prompt = "你好，这是我实操AI Infra的第一天！能给我一句简短的鼓励吗？"
encoded = tokenizer(prompt, return_tensors="pt") # 直接返回 Pytorch 张量格式（默认是 Pytorch list）
# 将输入数据移动到GPU上
encoded = encoded.to("cuda")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### 1.tokenizer

观察输入张量的维度和内容

In [2]:
print(f"encoded: {encoded}")
input_ids = encoded["input_ids"]
print(f"input_ids: {input_ids}")
# 观察输入张量的维度：[batch_size, sequence_length]
batch_size, prompt_len = input_ids.shape
print(f"输入张量shape：{input_ids.shape}")
print(f"token IDS: {input_ids[0].tolist()}")

encoded: {'input_ids': tensor([[108386,   3837, 108907,  39973, 100557,  15469,  14921,    956, 105525,
          35727,   6313,  26232, 104169, 104670,  98237,  99534,   9370, 104125,
         101037,  11319]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}
input_ids: tensor([[108386,   3837, 108907,  39973, 100557,  15469,  14921,    956, 105525,
          35727,   6313,  26232, 104169, 104670,  98237,  99534,   9370, 104125,
         101037,  11319]], device='cuda:0')
输入张量shape：torch.Size([1, 20])
token IDS: [108386, 3837, 108907, 39973, 100557, 15469, 14921, 956, 105525, 35727, 6313, 26232, 104169, 104670, 98237, 99534, 9370, 104125, 101037, 11319]


逐个观察：每个 ID 到底长什么样？

In [3]:
for token_id in input_ids[0]:
    # convert_ids_to_tokens: 看它在分词器词表的原始 key
    token_str = tokenizer.convert_ids_to_tokens(token_id.item())
    # decode: 看它被渲染成人类字符后的样子
    decode_char = tokenizer.decode([token_id.item()])
    print(f"token_id: {token_id.item():<6} -> key: {token_str:<10} -> char: {decode_char}")

token_id: 108386 -> key: ä½łå¥½     -> char: 你好
token_id: 3837   -> key: ï¼Į        -> char: ，
token_id: 108907 -> key: è¿Ļæĺ¯æĪĳ  -> char: 这是我
token_id: 39973  -> key: å®ŀ        -> char: 实
token_id: 100557 -> key: æĵį        -> char: 操
token_id: 15469  -> key: AI         -> char: AI
token_id: 14921  -> key: ĠInf       -> char:  Inf
token_id: 956    -> key: ra         -> char: ra
token_id: 105525 -> key: çļĦç¬¬ä¸Ģ  -> char: 的第一
token_id: 35727  -> key: å¤©        -> char: 天
token_id: 6313   -> key: ï¼ģ        -> char: ！
token_id: 26232  -> key: èĥ½        -> char: 能
token_id: 104169 -> key: ç»ĻæĪĳ     -> char: 给我
token_id: 104670 -> key: ä¸Ģåı¥     -> char: 一句
token_id: 98237  -> key: ç®Ģ        -> char: 简
token_id: 99534  -> key: çŁŃ        -> char: 短
token_id: 9370   -> key: çļĦ        -> char: 的
token_id: 104125 -> key: é¼ĵåĬ±     -> char: 鼓励
token_id: 101037 -> key: åĲĹ        -> char: 吗
token_id: 11319  -> key: ï¼Ł        -> char: ？


问：分词表里的词都是写好了的吗，同一个句子在同一个 tokenizer 上的分法是一致的吗？（个人看法：是）

问：分词对模型的理解有影响吗，是根据什么确定分词方法的，比如为什么要把“Infra”分为“Inf”+“ra”？（扩展内容）

问：目前市面上的主流旗舰模型应该都是支持全球各个常见语言的吧，那是否要在分词映射表中囊括每一个出现的字词？分词表是通用的还是对于不同的模型可能会有不同的分词表？

问：分词表是根据训练数据产生的吗？即拿一堆文本进行训练时，自然而然就出现了这些分词表。假设一家模型公司只有中英文语料，并进行训练，那模型是不是就不能理解德语之类的语言？甚至分词表都会没有相应的token id？

### 2.调用模型生成

In [4]:
print("开始生成")
outputs = model.generate(
    **encoded,               # 解包字典：传入 input_ids 和 attention_mask
    max_new_tokens=256,      # 纯增量生成 256 个新 token
    do_sample=False,         # 关闭随机采样，使用 Greedy 贪心解码（每次选 logit 最大的 token）
    repetition_penalty=1.0,  # 模型中的为 1.1
)

print(outputs)
total_len = outputs.shape[1]
gen_len = total_len - prompt_len

print(f"输出张量 shape: {outputs.shape}")
print(f"总token数：{total_len}；生成token数：{gen_len}")

开始生成
tensor([[108386,   3837, 108907,  39973, 100557,  15469,  14921,    956, 105525,
          35727,   6313,  26232, 104169, 104670,  98237,  99534,   9370, 104125,
         101037,  11319,  84897,  60894,  73670,   6313, 100549,  56568,  39973,
         100557,  15469,  14921,    956, 105525,  35727, 102088,   3837, 104177,
         101038, 103929, 100735,   6313, 102839,   6313, 102056, 110117,  86119,
          57191,  85106, 100364,   3837, 102422, 100437, 104150, 105396,   1773,
         100549,  56568, 100644, 107071,   6313, 151643]], device='cuda:0')
输出张量 shape: torch.Size([1, 60])
总token数：60；生成token数：40


In [5]:
generated_ids = outputs[0, prompt_len:]

print("\n增量生成内容:")
print(tokenizer.decode(generated_ids, skip_special_tokens=True))


增量生成内容:
 当然可以！祝你实操AI Infra的第一天顺利，期待看到你的成果！加油！如果你有任何问题或需要帮助，随时欢迎回来询问。祝你今天愉快！


## 二、看懂 Transformer 推理链路

> 学习内容：
> - embedding；
> - Transformer block；
> - RMSNorm；
> - self-attention；
> - MLP/SwiGLU；
> - LM head；
> - logits；
> - sampling。
> 不推公式，只关注张量如何流动

## 三、不用 generate()，手写 Decode Loop

第一个核心任务。
伪代码如下：
```python
input_ids = tokenizer(prompt).input_ids

for _ in range(max_new_tokens):
    outputs = model(input_ids=input_ids, use_cache=False)
    logits = outputs.logits[:, -1, :]
    next_token = logits.argmax(dim=-1, keepdim=True)
    input_ids = torch.cat([input_ids, next_token], dim=-1)
```
需要加入：
- model.eval()；
- torch.inference_mode()；
- EOS 判断；
- 固定 greedy decoding；
- 与 model.generate() 的结果对比。
注意：如果两者结果不一致，先检查：
- chat template；
- EOS token；
- generation config；
- attention mask；
- 是否使用 greedy decoding；
- 最大生成长度定义是否相同。
通关标准：
> 写的循环能够生成文本，并理解它为什么越来越慢：每轮都重新计算整个历史序列。

### 0.准备工作

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path

# 加载模型与tokenizer
MODEL_PATH = Path("../../models/Qwen2.5-1.5B-Instruct")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="cuda"
)

# 设置推理模式与评估模式（如会关闭dropout、冻结BatchNorm）
model.eval()

# 准备测试文本及分词
prompt = "你好，这是我实操AI Infra的第一天！能给我一句简短的鼓励吗？"
encoded = tokenizer(prompt, return_tensors="pt") # 直接返回 Pytorch 张量格式（默认是 Pytorch list）
# 将输入数据移动到GPU上
encoded = encoded.to("cuda")

input_ids = encoded["input_ids"]
max_new_tokens = 256
eos_token_id = tokenizer.eos_token_id

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

151645


### 1. 手写 Decode Loop（禁用 kv cache）

In [10]:
generated_ids = input_ids.clone()

with torch.inference_mode(): # 禁用梯度与追踪，极大提升推理效率并节省缓存
    for step in range(max_new_tokens):
        # 前向传播（显式关闭 kv cache）
        outputs = model(input_ids=generated_ids, use_cache=False)

        # 提取最后一个位置的 Logits：[B, T, V] -> [B, V]
        next_token_logits = outputs.logits[:, -1, :]

        # 贪婪采样（Greedy decoding）：选概率最高的 token
        next_token = next_token_logits.argmax(dim=-1, keepdim=True)

        # 拼接到历史序列末尾
        generated_ids = torch.cat([generated_ids, next_token], dim=-1)

        # EOS 判断（仅在 batch 为 1 时可以写成如下形式）
        # if next_token.item() == eos_token_id: # 模型配置中是 [151645, 151643]，而获取到的 eos_token_id 会是 151645，不符合
        if next_token.item() in [151645, 151643]:
            print(f"触发EOS，在第 {step + 1} 步停止生成")
            break

触发EOS，在第 40 步停止生成


### 2. 解码输出文本

In [11]:
custom_output_text = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
)

print("手写Loop输出结果：", custom_output_text)

手写Loop输出结果： 你好，这是我实操AI Infra的第一天！能给我一句简短的鼓励吗？ 当然可以！祝你实操AI Infra的第一天顺利，期待看到你的成果！加油！如果你有任何问题或需要帮助，随时欢迎回来询问。祝你今天愉快！


## 四、 带 KV Cache 的手写 Decode Loop

In [21]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path

# 加载模型与tokenizer
MODEL_PATH = Path("../../models/Qwen2.5-1.5B-Instruct")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="cuda"
)

# 设置推理模式与评估模式（如会关闭dropout、冻结BatchNorm）
model.eval()

# 准备测试文本及分词
prompt = "你好，这是我实操AI Infra的第一天！能给我一句简短的鼓励吗？"
encoded = tokenizer(prompt, return_tensors="pt") # 直接返回 Pytorch 张量格式（默认是 Pytorch list）
# 将输入数据移动到GPU上
encoded = encoded.to("cuda")

input_ids = encoded["input_ids"]
max_new_tokens = 256
eos_token_idS = [151645, 151643]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### 1.预填充阶段 -- 处理完整 Prompt， 构建初始 KV Cache

In [28]:
with torch.inference_mode():
    outputs = model(input_ids=input_ids, use_cache=True)

    # 当前 Transformers 返回 DynamicCache，直接访问每层的 Key/Value
    past_key_values = outputs.past_key_values

    # 获取第一个生成的 token
    next_token_logits = outputs.logits[:, -1, :]
    next_token = next_token_logits.argmax(dim=-1, keepdim=True)

    # 记录生成的 token ID 列表
    cache_generated_ids = [next_token.item()]

    # 打印第一层 KV Cache 的维度形状
    first_layer_cache = past_key_values.layers[0]
    first_layer_k = first_layer_cache.keys
    first_layer_v = first_layer_cache.values

    print("[Prefill 阶段完成]")
    print(f"Prompt 长度： {input_ids.shape[-1]}")
    print(f"第 1 层 Key Cache 形状：{first_layer_k.shape}")
    print(f"第 1 层 Value Cache 形状：{first_layer_v.shape}")

    current_input_ids = next_token # 输入张量形状为 [B, 1]

    print("Decode 阶段开始")
    for step in range(max_new_tokens):
        # 传入当前的 1 个 token 以及过去的 past_key_values
        outputs = model(
            input_ids=current_input_ids,
            past_key_values=past_key_values,
            use_cache=True,
        )

        # 拿到更新后的 kv cache
        past_key_values = outputs.past_key_values

        next_token_logits = outputs.logits[:, -1, :]
        next_token = next_token_logits.argmax(dim=-1, keepdim=True)

        token_id = next_token.item()
        cache_generated_ids.append(token_id)

        # 更新下一轮的输入 token
        current_input_ids = next_token

        # 打印当前步骤 kv cache 维度变化
        print(f"Step {step+1}：输入 input_ids 形状：{current_input_ids.shape}, key cache 形状：{past_key_values.layers[0].keys.shape}")

        if token_id in eos_token_idS:
            print("[触发 EOS 停止]")
            break

[Prefill 阶段完成]
Prompt 长度： 20
第 1 层 Key Cache 形状：torch.Size([1, 2, 20, 128])
第 1 层 Value Cache 形状：torch.Size([1, 2, 20, 128])
Decode 阶段开始
Step 1：输入 input_ids 形状：torch.Size([1, 1]), key cache 形状：torch.Size([1, 2, 21, 128])
Step 2：输入 input_ids 形状：torch.Size([1, 1]), key cache 形状：torch.Size([1, 2, 22, 128])
Step 3：输入 input_ids 形状：torch.Size([1, 1]), key cache 形状：torch.Size([1, 2, 23, 128])
Step 4：输入 input_ids 形状：torch.Size([1, 1]), key cache 形状：torch.Size([1, 2, 24, 128])
Step 5：输入 input_ids 形状：torch.Size([1, 1]), key cache 形状：torch.Size([1, 2, 25, 128])
Step 6：输入 input_ids 形状：torch.Size([1, 1]), key cache 形状：torch.Size([1, 2, 26, 128])
Step 7：输入 input_ids 形状：torch.Size([1, 1]), key cache 形状：torch.Size([1, 2, 27, 128])
Step 8：输入 input_ids 形状：torch.Size([1, 1]), key cache 形状：torch.Size([1, 2, 28, 128])
Step 9：输入 input_ids 形状：torch.Size([1, 1]), key cache 形状：torch.Size([1, 2, 29, 128])
Step 10：输入 input_ids 形状：torch.Size([1, 1]), key cache 形状：torch.Size([1, 2, 30, 128])
Step 11：输入 input_ids 形

In [30]:
token = tokenizer.decode(cache_generated_ids, skip_special_tokens=True)
print(token)

 当然可以！祝你实操AI Infra的第一天顺利，期待看到你的成果！加油！如果你有任何问题或需要帮助，随时欢迎回来询问。祝你今天愉快！


## 五、理解 Prefill 与 Decode

Prefill

输入完整 prompt：

[B, T] → 模型 → [B, T, vocab]

特点：
- 一次处理很多 token；
- 大量矩阵乘法；
- token 维度存在并行性；
- 通常具有较高计算密度；
- 容易更充分利用 GPU。

Decode

每次只加入一个 token：

[B, 1] + KV Cache → 下一个token

特点：

- 每步只生成一个 token；
- 重复读取大量模型权重；
- 需要访问不断增长的 KV Cache；
- 单请求下矩阵形状较小；
- GPU 可能吃不满；
- 通常更容易受内存带宽和调度开销限制。

但是注意：

“Prefill 一定 compute-bound、Decode 一定 memory-bound”只是常见情况，不是物理定律。模型结构、batch、硬件、量化和上下文长度都会改变瓶颈。

任务：手算两个数量。

模型权重大小

粗略估计：参数量 × 每参数字节数

例如 0.5B 参数、BF16：0.5 × 10^9 × 2 bytes ≈ 1 GB

KV Cache 大小

近似计算：
2 × 层数 × batch
  × KV heads × sequence length
  × head dimension × 每元素字节数
前面的 2 是 K 和 V。

通关标准：能根据 batch 和 sequence length 的变化，判断 KV Cache 大小如何变化。

## 六、验收清单

- [ ] 能运行小模型；
- [ ] 能手写 greedy decoding；
- [ ] 能加入 KV Cache；
- [ ] 两种实现输出一致；
- [ ] 能打印和解释主要张量 shape；
- [ ] 能区分 Prefill 与 Decode；
- [ ] 能粗算模型和 KV Cache 显存。